# UI Critic — Pilot Test
Pipeline'ın GPU'da hatasız çalıştığını doğrular. Gerçek RICO yerine sahte görsel kullanır, sadece 5 adım eğitir.

In [ ]:
# Hücre 1: GPU kontrolü
!nvidia-smi

In [ ]:
# Hücre 2: Gereksinimler
!pip install -q transformers>=4.46 peft>=0.13 bitsandbytes>=0.44 accelerate pillow pandas pyyaml pydantic tqdm gdown huggingface-hub

In [ ]:
# Hücre 3: Repo klonla
!git clone https://github.com/umutakbass/ui-critic-project.git
import os
os.chdir('ui-critic-project')
!ls

In [ ]:
# Hücre 4: UICrit CSV indir (Drive'dan)
!mkdir -p data/uicrit
!gdown 1FcewVy9PJ5RiqC6L1FjICdY4VJoMvuPf -O data/uicrit/uicrit.zip
!unzip -q data/uicrit/uicrit.zip -d data/uicrit/
!rm data/uicrit/uicrit.zip
!ls data/uicrit/

In [ ]:
# Hücre 5: Sahte RICO görselleri oluştur (pipeline testi için, gerçek indirmeye gerek yok)
import sys, json, random
from pathlib import Path
from PIL import Image, ImageDraw

sys.path.insert(0, 'src')
from data.uicrit_loader import UICritLoader

loader = UICritLoader('data/uicrit/uicrit_public.csv')
df = loader.load()
rico_ids = df['rico_id'].unique()[:50]  # sadece 50 ID yeterli

out_dir = Path('data/archive/unique_uis/combined')
out_dir.mkdir(parents=True, exist_ok=True)

for rid in rico_ids:
    # Sahte UI görseli
    img = Image.new('RGB', (1080, 1920), color=(245, 245, 245))
    draw = ImageDraw.Draw(img)
    for _ in range(8):
        x1, y1 = random.randint(0, 900), random.randint(0, 1800)
        draw.rectangle([x1, y1, x1+150, y1+80],
                       fill=(random.randint(100,220), random.randint(100,220), random.randint(100,220)))
    img.save(out_dir / f'{rid}.jpg')

    # Sahte view hierarchy
    vh = {
        'activity_name': 'com.test.MainActivity',
        'activity': {'root': {
            'class': 'android.widget.FrameLayout',
            'bounds': [0, 0, 1080, 1920],
            'visible-to-user': True,
            'children': [
                {'class': 'android.widget.TextView', 'bounds': [50, 100, 500, 180],
                 'text': 'Başlık', 'visible-to-user': True, 'children': []},
                {'class': 'android.widget.Button', 'bounds': [400, 1700, 900, 1820],
                 'text': 'Devam', 'clickable': True, 'visible-to-user': True, 'children': []}
            ]
        }},
        'is_keyboard_deployed': False,
        'request_id': str(rid)
    }
    with open(out_dir / f'{rid}.json', 'w') as f:
        json.dump(vh, f)

print(f'✓ {len(rico_ids)} sahte UI oluşturuldu → {out_dir}')

In [ ]:
# Hücre 6: Eğitim verisi hazırla
!python scripts/prepare_data.py --task model1
!ls data/processed/

In [ ]:
# Hücre 7: HuggingFace login
# Token: huggingface.co → Settings → Access Tokens → New token (Read)
from huggingface_hub import login
login()  # token soracak, yapıştır

In [ ]:
# Hücre 8: Pilot eğitim — 5 adım, Qwen2.5-VL-3B
# Model indirme ~5 dakika, eğitim ~2 dakika
from training.config_loader import load_config, apply_overrides
from training.trainer import train

config = load_config('configs/model1_qwen.yaml')
config = apply_overrides(config, [
    'model.name=qwen2.5-vl-3b',  # 7B yerine 3B, Colab'a sığar
    'training.max_steps=5',       # sadece 5 adım
    'training.batch_size=1',
    'training.gradient_accumulation_steps=1',
    'training.logging_steps=1',
    'training.save_steps=10',
    'training.eval_steps=10',
])

print(f'Model: {config.model.name}')
print(f'Max steps: {config.training.max_steps}')
train(config)

## Sonuç
Son hücre hatasız bittiyse pipeline tamam demektir:
- Model yüklendi ✓
- LoRA uygulandı ✓  
- Forward pass çalıştı ✓
- Loss hesaplandı ✓

Artık RunPod'da gerçek eğitimi başlatabilirsin.